# Paper 2: from-scratch gravitational attention (Colab)
Runtime → Change runtime type → A100 GPU. Upload `gravitational-inductive-bias.zip` to `MyDrive/GAM_Paper2/` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/GAM_Paper2'

In [ ]:
!mkdir -p /content/code && cd /content/code && unzip -o -q $ROOT/gravitational-inductive-bias.zip
%cd /content/code/gravitational-inductive-bias
!pip install -q -r requirements.txt

In [ ]:
!python -m pytest -q tests

## Pilot (8 short runs, seed 99, 2,000 steps each)
Safe to re-run after a disconnect: finished runs are skipped, an interrupted run restarts.

In [ ]:
!python scripts/run_matrix.py pilot --root $ROOT/pilot --extra "--amp --cache_dir $ROOT/data_cache"

In [ ]:
import json, glob, statistics as st
for f in sorted(glob.glob(f'{ROOT}/pilot/*/results.json')):
    r = json.load(open(f))
    print(f.split('/')[-2].ljust(26), 'val loss %.3f' % r['final']['val_loss'],
          '| %.0f s' % r['train_time_s'], '| %.0f MB' % (r['peak_mem_mb'] or 0), '|', r['counters'])

## Confirmatory runs (only after the pilot decisions are fixed in configs/protocol.yaml)

In [ ]:
# !python scripts/run_matrix.py A --root $ROOT/runs --extra "--amp --cache_dir $ROOT/data_cache"

## Experiment C: 512-token sequences (added 23 Nov 2026)
First run the short pilot (2 runs of 300 steps) to measure time and memory. The first run also builds the 512-token data cache, which takes a few extra minutes.

In [ ]:
!python scripts/run_matrix.py pilot_C --root $ROOT/pilot_C --extra "--amp --cache_dir $ROOT/data_cache"

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{ROOT}/pilot_C/*/results.json')):
    r = json.load(open(f))
    s_per_step = r['train_time_s'] / r['args']['steps']
    print(f.split('/')[-2].ljust(22), r['env'].get('gpu', ''), '| %.2f s/step' % s_per_step,
          '| full run about %.1f h' % (s_per_step * 20000 / 3600), '| peak %.0f MB' % (r['peak_mem_mb'] or 0), '|', r['counters'])

## Experiment C: full runs (9 runs: A1, A2, A4 x seeds 0-2)
Safe to re-run after a disconnect: finished runs are skipped, an interrupted run restarts.

In [ ]:
!python scripts/run_matrix.py C --root $ROOT/runs --extra "--amp --cache_dir $ROOT/data_cache"